Data Lineage Parsing – Status Update
Plan of Action (High-Level Design)
I structured the data lineage pipeline into nine layers:
Input Layer – File discovery & ingestion from repo.


Parsing Layer – SQL, Python (ORM), JS/TS (API calls), config parsing.


Entity Extraction Layer – Normalize & classify entities (tables, views, columns, procedures, APIs).


Relationship Mapping Layer – Build edges (joins, CRUD, filters, procedure–table, API–procedure, dashboard–view).


Operation Extraction Layer – Detect CRUD, DDL, transformations, aggregations.


NLP Enrichment Layer – Summarize entities & stored procedures.


Graph Model Layer – Persist entities, relationships, and operations into a registry.


Output Layer – Export lineage (Snowflake, Neo4j, NetworkX, CSV/GraphML).


Orchestration Layer – Prefect flow for scheduling, monitoring, retries, and alerting.


INPUT MODULE: PARSES frontend repo(guided-workflow) and backend repo(guded-workflow-backend) and returns files{<file_path>:<file content>}

In [ ]:
import os
import chardet
from typing import List, Dict, Generator


class InputLayer:
    """
    Input layer to collect and tokenize files from a codebase.
    Uses chardet for safe encoding detection.
    """
    def __init__(self, base_path: str, allowed_ext: List[str] = None, ignore_dirs: List[str] = None):
        self.base_path = base_path
        self.allowed_ext = allowed_ext or [".sql", ".py", ".ts", ".js"]
        self.ignore_dirs = set(ignore_dirs or ["__pycache__", ".git", "node_modules", "dist", "build", "venv"])

    def walk_directory(self) -> Generator[str, None, None]:
        """Traverse repo and yield files with allowed extensions, skipping ignored dirs."""
        for root, dirs, files in os.walk(self.base_path):
            # Modify dirs in-place → prevents os.walk from descending into ignored ones
            dirs[:] = [d for d in dirs if d not in self.ignore_dirs]

            for file in files:
                if not any(file.endswith(ext) for ext in self.allowed_ext):
                    continue
                yield os.path.join(root, file)

    def read_file(self, file_path: str) -> str:
        """Read file with encoding detection."""
        try:
            with open(file_path, "rb") as f:
                raw_data = f.read()
                encoding = chardet.detect(raw_data)["encoding"] or "utf-8"
            return raw_data.decode(encoding, errors="ignore")
        except Exception as e:
            print(f" Error reading {file_path}: {e}")
            return ""

    def tokenize_file(self, content: str, file_path: str) -> Dict:
        """Return file-level tokens (not line-based)."""
        return {
            "file_path": file_path,
            "content": content
        }

    def ingest(self) -> List[Dict]:
        """Collect all repo files as {path, content} dicts."""
        all_files = []
        for file_path in self.walk_directory():
            content = self.read_file(file_path)
            if content.strip():
                all_files.append(self.tokenize_file(content, file_path))
        return all_files


# Example run
if __name__ == "__main__":
    base_repo = "C:/dev/guided-workflow-backend"
  # replace with your repo path
    ingestor = InputLayer(base_repo, ignore_dirs=["__pycache__",".venv", ".git", "dist", "build",".env",".gitattributes",".gitignore","node_modules", "tests"])
    files = ingestor.ingest()

  

    print(f" Total files ingested: {len(files)}")
    print(" Sample:", files[0] if files else "No files found")

    




DATA MODEL : we are using dataclasses and defining data model to catpure parses data in the below data classes

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import List
import uuid


@dataclass
class Entity:
    entity_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    entity_name: str = ""
    entity_type: str = ""
    qualified_name: str = ""
    parent_entity_id: Optional[str] = None
    description: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)
    created_at: datetime = field(default_factory=datetime.utcnow)
    updated_at: datetime = field(default_factory=datetime.utcnow)

@dataclass
class RelationshipType:
    relationship_type_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    name: str = ""
    category: str = ""  
    description: str = ""
    created_at: datetime = field(default_factory=datetime.utcnow)

@dataclass
class Relationship:
    relationship_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    source_entity_id: str = ""
    target_entity_id: str = ""
    relationship_type_id: str = ""
    relationship_detail: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)
    created_at: datetime = field(default_factory=datetime.utcnow)

@dataclass
class Operation:
    operation_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    operation_type: str = ""
    source_entity_id: str = ""
    target_entity_id: Optional[str] = None
    performed_at: datetime = field(default_factory=datetime.utcnow)
    performed_by: str = ""
    sql_snippet: str = ""
    execution_context: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

# Complete set of relationship types
relationship_types: List[RelationshipType] = [
    # Existing 25 types
    RelationshipType("r001", "CONTAINS_COLUMN", "STRUCTURE", "Table contains a column"),
    RelationshipType("r002", "IS_PRIMARY_KEY", "STRUCTURE", "Column is part of primary key"),
    RelationshipType("r003", "IS_FOREIGN_KEY", "STRUCTURE", "Column references another table"),
    RelationshipType("r004", "IS_NULLABLE", "STRUCTURE", "Column allows null values"),
    RelationshipType("r005", "HAS_INDEX", "STRUCTURE", "Column has database index"),
    RelationshipType("r006", "HAS_CONSTRAINT", "STRUCTURE", "Column has check constraint"),
    RelationshipType("r007", "USES_TABLE", "USAGE", "Procedure reads from table"),
    RelationshipType("r008", "USED_AS_SELECT", "USAGE", "Column selected in query"),
    RelationshipType("r009", "USED_AS_JOIN", "USAGE", "Column used for table join"),
    RelationshipType("r010", "USED_AS_FILTER", "USAGE", "Column used in WHERE clause"),
    RelationshipType("r011", "USED_AS_GROUP_BY", "USAGE", "Column used in GROUP BY clause"),
    RelationshipType("r012", "USED_AS_ORDER_BY", "USAGE", "Column used in ORDER BY clause"),
    RelationshipType("r013", "USED_AS_HAVING", "USAGE", "Column used in HAVING clause"),
    RelationshipType("r014", "IS_TRANSFORMED_AS", "TRANSFORMATION", "Column undergoes data transformation"),
    RelationshipType("r015", "DERIVED_FROM", "TRANSFORMATION", "Field calculated from other fields"),
    RelationshipType("r016", "AGGREGATED_FROM", "TRANSFORMATION", "Field is aggregation of other fields"),
    RelationshipType("r017", "PROCEDURE_CREATES_TABLE", "EXECUTION", "Procedure creates table"),
    RelationshipType("r018", "PROCEDURE_DROPS_TABLE", "EXECUTION", "Procedure drops table"),
    RelationshipType("r019", "BACKEND_COMMUNICATES_TABLE", "EXECUTION", "Backend communicates table"),
    RelationshipType("r020", "PROCEDURE_CALLS_PROCEDURE", "EXECUTION", "Backend service calls procedure"),
    RelationshipType("r021", "BACKEND_CALLS_FUNCTION", "EXECUTION", "Backend calls function"),
    RelationshipType("r022", "CTE_REFERENCES", "USAGE", "CTE references table"),
    RelationshipType("r023", "UNION_WITH", "USAGE", "Table/CTE combined with UNION"),
    RelationshipType("r024", "LEFT_JOIN_WITH", "USAGE", "Left join relationship"),
    RelationshipType("r025", "INNER_JOIN_WITH", "USAGE", "Inner join relationship"),
    
    # Additional join types
    RelationshipType("r026", "RIGHT_JOIN_WITH", "USAGE", "Right join relationship"),
    RelationshipType("r027", "FULL_OUTER_JOIN_WITH", "USAGE", "Full outer join relationship"),
    RelationshipType("r028", "CROSS_JOIN_WITH", "USAGE", "Cross join relationship"),
    RelationshipType("r029", "SELF_JOIN_WITH", "USAGE", "Self join relationship"),
    
    # SQL transformations & derived data
    RelationshipType("r030", "DERIVED_FROM_TABLE", "TRANSFORMATION", "Table derived from another table/query"),
    RelationshipType("r031", "TEMP_TABLE_USED_BY", "USAGE", "Temporary/transient table used by another entity"),
    RelationshipType("r032", "COLUMN_HAS_DEFAULT", "STRUCTURE", "Column has default value"),
    RelationshipType("r033", "COLUMN_USED_IN_CASE", "USAGE", "Column used in CASE expression"),
    RelationshipType("r034", "COLUMN_USED_IN_COALESCE", "USAGE", "Column used in COALESCE/NVL/DECODE"),
    RelationshipType("r035", "COLUMN_PART_OF_WINDOW_FUNCTION", "USAGE", "Column used in window/analytical function"),
    RelationshipType("r036", "COLUMN_PART_OF_SET_OPERATION", "USAGE", "Column used in UNION/INTERSECT/EXCEPT"),
    
    # Procedure operations
    RelationshipType("r037", "PROCEDURE_UPDATES_TABLE", "EXECUTION", "Procedure updates a table"),
    RelationshipType("r038", "PROCEDURE_READS_TABLE", "USAGE", "Procedure reads from a table"),
    RelationshipType("r039", "PROCEDURE_CALLS_FUNCTION", "EXECUTION", "Procedure calls a function"),
    RelationshipType("r040", "PROCEDURE_USES_TEMP_TABLE", "USAGE", "Procedure uses temp/transient table"),
    RelationshipType("r041", "PROCEDURE_HAS_AGGREGATE", "TRANSFORMATION", "Procedure aggregates data using SUM, AVG, etc."),
    RelationshipType("r042", "PROCEDURE_USES_WINDOW_FUNCTION", "TRANSFORMATION", "Procedure uses window/analytical functions"),
    
    # Python ORM
    RelationshipType("r043", "PYTHON_EXECUTES_QUERY", "EXECUTION", "Python executes SQL using cursor or connection"),
    RelationshipType("r044", "PYTHON_USES_ORM_TABLE", "USAGE", "Python ORM accesses a table"),
    RelationshipType("r045", "PYTHON_USES_ORM_COLUMN", "USAGE", "Python ORM accesses a column"),
    RelationshipType("r046", "PYTHON_CALLS_PROCEDURE", "EXECUTION", "Python calls stored procedure via DB API"),
    RelationshipType("r047", "PYTHON_TRANSFORMS_COLUMN", "TRANSFORMATION", "Python transforms a column before writing"),
    
    # Frontend / API
    RelationshipType("r048", "API_CALLS_BACKEND", "EXECUTION", "Frontend API calls backend function/service"),
    RelationshipType("r049", "API_CONSUMES_ENTITY", "USAGE", "API consumes data from table/procedure"),
    RelationshipType("r050", "API_UPDATES_ENTITY", "EXECUTION", "API updates data in table/procedure"),
    RelationshipType("r051", "BACKEND_TRIGGER_PROCEDURE", "EXECUTION", "Backend triggers a stored procedure"),
    RelationshipType("r052", "BACKEND_UPDATES_ENTITY", "EXECUTION", "Backend updates table/view"),
    RelationshipType("r053", "BACKEND_READS_ENTITY", "USAGE", "Backend reads table/view"),
    
    # Config / metadata
    RelationshipType("r054", "CONFIG_REFERENCES_ENTITY", "USAGE", "Config file references table/view/schema"),
    RelationshipType("r055", "CONFIG_USED_FOR_CONNECTION", "USAGE", "Config used to connect to DB/warehouse"),
    
    # General / derived
    RelationshipType("r056", "DERIVED_FIELD_FROM_COLUMNS", "TRANSFORMATION", "Column derived from other columns"),
    RelationshipType("r057", "DERIVED_TABLE_FROM_COLUMNS", "TRANSFORMATION", "Table derived from columns of another table"),
    RelationshipType("r058", "ALIAS_OF", "USAGE", "Column or table alias"),
    RelationshipType("r059", "RENAMED_AS", "TRANSFORMATION", "Table or column renamed"),
]


PARSE MODULE:Idea was to handle sql queries seprate and handle pyhton and typescript files seprate, For SQL I am using SQLGLOT package (but found out it has its own limitations) take files as input from input layer and parse the code to pull out SQL statement(SPs,VIEWS,TABLES)

PARSING FRONTEND and BACKEND: this gave me decent results in mapping Frontend action function calls to backend APIs was able to get close to 50% accuracy used libcst,tree_sitter

In [2]:
import os
import re
import json
import libcst as cst
from tree_sitter import Language, Parser
import tree_sitter_typescript as ts_typescript
from typing import List, Dict, Any

# ----------------------------
# CONFIG
# ----------------------------
IGNORE_DIRS = {"__pycache__", ".venv", ".git", "dist", "build", "node_modules", "tests"}

# ----------------------------
# FRONTEND PARSER (Fixed)
# ----------------------------
TS_LANGUAGE = Language(ts_typescript.language_typescript())
ts_parser = Parser(TS_LANGUAGE)

def normalize_url(url: str) -> str:
    """Replace dynamic path params with {param} for fuzzy matching"""
    if not url:
        return ""
    
    # Remove quotes and backticks
    url = url.strip('"\'`')
    
    # Handle template literals: ${V2_URL}/announcements -> /announcements
    url = re.sub(r'\$\{[^}]+\}', '', url)
    
    # Handle path parameters: /announcements/{id} -> /announcements/{param}
    url = re.sub(r'\{[^}]+\}', '{param}', url)
    
    # Handle variable interpolation: /announcements/${id} -> /announcements/{param}
    url = re.sub(r'/\$\{[^}]+\}', '/{param}', url)
    
    # Clean up multiple slashes and trailing slashes
    url = re.sub(r'/+', '/', url)
    url = url.rstrip('/')
    
    # Ensure starts with /
    if url and not url.startswith('/'):
        url = '/' + url
        
    return url

def extract_url_from_template_literal(template_str: str) -> str:
    """Extract URL from template literal like `${V2_URL}/announcements`"""
    template_str = template_str.strip('`')
    template_str = re.sub(r'\$\{V2_URL\}', '', template_str)
    template_str = re.sub(r'\$\{[^}]+\}', '{param}', template_str)
    return template_str

def extract_url_from_args(args_node, source_code) -> str:
    """Extract URL from function arguments"""
    if not args_node or len(args_node.children) == 0:
        return ""
    
    first_arg = None
    for child in args_node.children:
        if child.type not in ("(", ")", ","):
            first_arg = child
            break
    
    if not first_arg:
        return ""
    
    url_text = source_code[first_arg.start_byte:first_arg.end_byte].decode()
    
    if first_arg.type == "template_string":
        return extract_url_from_template_literal(url_text)
    elif first_arg.type == "string":
        return url_text.strip('"\'')
    else:
        return url_text

def find_function_context(node, source_code):
    """Find the function name that contains this node"""
    current = node.parent
    
    while current:
        if current.type == "variable_declarator":
            name_node = current.child_by_field_name("name")
            value_node = current.child_by_field_name("value")
            
            if name_node and value_node:
                if value_node.type == "arrow_function" or (
                    value_node.type == "call_expression" and 
                    len(value_node.children) > 0 and 
                    "async" in source_code[value_node.start_byte:value_node.end_byte].decode()
                ):
                    return source_code[name_node.start_byte:name_node.end_byte].decode()
        
        elif current.type == "function_declaration":
            name_node = current.child_by_field_name("name")
            if name_node:
                return source_code[name_node.start_byte:name_node.end_byte].decode()
        
        current = current.parent
    
    return None

def traverse_ts_fixed(node, source_code, results=None):
    """Fixed traverse function that properly finds function context"""
    if results is None:
        results = []

    if node.type == "call_expression":
        func_node = node.child_by_field_name("function")
        if func_node and func_node.type == "member_expression":
            obj_node = func_node.child_by_field_name("object")
            prop_node = func_node.child_by_field_name("property")
            
            if obj_node and prop_node:
                obj_name = source_code[obj_node.start_byte:obj_node.end_byte].decode()
                prop_name = source_code[prop_node.start_byte:prop_node.end_byte].decode()

                if obj_name == "client":
                    function_name = find_function_context(node, source_code)
                    
                    if function_name:
                        args_node = node.child_by_field_name("arguments")
                        args_text = ""
                        url = ""
                        
                        if args_node:
                            args_text = source_code[args_node.start_byte:args_node.end_byte].decode()
                            
                            if prop_name in ["get", "post", "put", "delete", "patch"]:
                                url = extract_url_from_args(args_node, source_code)

                        results.append({
                            "action_function": function_name,
                            "frontend_service": f"{obj_name}.{prop_name}",
                            "frontend_args": args_text,
                            "url": normalize_url(url)
                        })

    for child in node.children:
        traverse_ts_fixed(child, source_code, results)

    return results

def parse_frontend_fixed(root_dir: str):
    """Parse all frontend files with fixed logic"""
    results = []
    
    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames[:] = [d for d in dirnames if d not in IGNORE_DIRS]
        
        for file in filenames:
            if file.endswith((".ts", ".tsx", ".js")):
                path = os.path.join(dirpath, file)
                try:
                    with open(path, "rb") as f:
                        source_code = f.read()
                    tree = ts_parser.parse(source_code)
                    file_results = traverse_ts_fixed(tree.root_node, source_code)
                    results.extend(file_results)
                except Exception as e:
                    print(f"Error parsing {path}: {e}")
    
    return results

# ----------------------------
# BACKEND PARSER (Fixed with Prefixes)
# ----------------------------

def extract_router_prefixes_simple(init_file_path: str) -> Dict[str, str]:
    """Extract router prefixes using simple string parsing"""
    file_to_prefix = {}
    
    try:
        with open(init_file_path, 'r', encoding='utf8') as f:
            content = f.read()
        
        import_pattern = r'from\s+\.(\w+)\s+import\s+router\s+as\s+(\w+)'
        imports = re.findall(import_pattern, content)
        
        router_to_file = {router_alias: file_name for file_name, router_alias in imports}
        
        include_pattern = r'router\.include_router\(\s*(\w+),\s*prefix=["\']([^"\']+)["\']'
        includes = re.findall(include_pattern, content)
        
        for router_alias, prefix in includes:
            if router_alias in router_to_file:
                file_name = router_to_file[router_alias]
                file_to_prefix[file_name] = prefix
        
        return file_to_prefix
        
    except Exception as e:
        print(f"Error extracting router prefixes from {init_file_path}: {e}")
        return {}

class FastAPIVisitorWithPrefix(cst.CSTVisitor):
    """Visitor to extract FastAPI routes with router context"""
    
    def __init__(self, file_name: str, router_prefix: str = ""):
        self.endpoints = []
        self.file_name = file_name
        self.router_prefix = router_prefix.rstrip('/')

    def visit_FunctionDef(self, node: cst.FunctionDef) -> None:
        """Visit function definitions to find route decorators"""
        if not node.decorators:
            return

        for decorator in node.decorators:
            try:
                decorator_code = cst.Module([]).code_for_node(decorator.decorator)
                
                if decorator_code.startswith("router."):
                    method_and_route = decorator_code[7:]
                    
                    if "(" in method_and_route:
                        method = method_and_route.split("(")[0].upper()
                        route_part = method_and_route.split("(", 1)[1]
                        
                        route = self._extract_route_from_args(route_part)
                        full_route = self._combine_prefix_and_route(self.router_prefix, route)
                        
                        self.endpoints.append({
                            "function": node.name.value,
                            "method": method,
                            "route": self._normalize_route(full_route),
                            "file": self.file_name,
                            "router_prefix": self.router_prefix,
                            "local_route": route
                        })
            except Exception:
                continue

    def _extract_route_from_args(self, args_str: str) -> str:
        """Extract route from decorator arguments"""
        args_str = args_str.rstrip(")")
        
        if args_str.strip() == '""' or args_str.strip() == "''":
            return ""
        
        if args_str.startswith('"') or args_str.startswith("'"):
            quote_char = args_str[0]
            end_quote = args_str.find(quote_char, 1)
            if end_quote != -1:
                return args_str[1:end_quote]
        
        if "," in args_str:
            first_arg = args_str.split(",")[0].strip()
            return first_arg.strip('"\'')
        
        return args_str.strip('"\'')
    
    def _combine_prefix_and_route(self, prefix: str, route: str) -> str:
        """Combine router prefix with local route"""
        if not prefix:
            return route if route else "/"
        
        if not route or route == "":
            return prefix
        
        if not prefix.startswith('/'):
            prefix = '/' + prefix
        
        if route.startswith('/'):
            return prefix + route
        else:
            return prefix + '/' + route
    
    def _normalize_route(self, route: str) -> str:
        """Normalize route for consistent formatting"""
        if not route:
            return "/"
        
        if not route.startswith('/'):
            route = '/' + route
        
        route = re.sub(r'\{[^}]+\}', '{param}', route)
        route = re.sub(r'/+', '/', route)
        
        if route != '/' and route.endswith('/'):
            route = route.rstrip('/')
        
        return route

def parse_backend_with_prefixes_fixed(root_dir: str):
    """Parse all backend files considering router prefixes"""
    backend_endpoints = []
    
    router_init_path = os.path.join(root_dir, "v2", "routers", "__init__.py")
    router_prefixes = {}
    
    if os.path.exists(router_init_path):
        router_prefixes = extract_router_prefixes_simple(router_init_path)
    
    routers_dir = os.path.join(root_dir, "v2", "routers")
    
    if os.path.exists(routers_dir):
        for file in os.listdir(routers_dir):
            if file.endswith(".py") and file != "__init__.py":
                file_path = os.path.join(routers_dir, file)
                file_name = file.replace('.py', '')
                prefix = router_prefixes.get(file_name, "")
                
                try:
                    with open(file_path, encoding="utf8") as f:
                        content = f.read()
                    
                    module = cst.parse_module(content)
                    visitor = FastAPIVisitorWithPrefix(file_name, prefix)
                    module.visit(visitor)
                    backend_endpoints.extend(visitor.endpoints)
                except Exception as e:
                    print(f"Error parsing {file_path}: {e}")
    
    return backend_endpoints

# ----------------------------
# IMPROVED MAPPING LOGIC
# ----------------------------

def calculate_route_similarity(frontend_url: str, backend_route: str) -> float:
    """Calculate similarity between frontend URL and backend route"""
    if not frontend_url or not backend_route:
        return 0.0
    
    if frontend_url == backend_route:
        return 1.0
    
    frontend_segments = [s for s in frontend_url.split('/') if s]
    backend_segments = [s for s in backend_route.split('/') if s]
    
    if len(frontend_segments) != len(backend_segments):
        return 0.0
    
    matches = 0
    for f_seg, b_seg in zip(frontend_segments, backend_segments):
        if f_seg == b_seg:
            matches += 1
        elif f_seg == "{param}" or b_seg == "{param}":
            matches += 0.8
    
    return matches / len(frontend_segments) if frontend_segments else 0.0

def map_frontend_to_backend_final(frontend_calls: List[Dict], backend_endpoints: List[Dict]) -> List[Dict]:
    """Map frontend calls to backend endpoints with improved logic"""
    mapping = []
    
    for call in frontend_calls:
        if not call.get('url'):
            continue
            
        frontend_method = call['frontend_service'].split('.')[-1].upper()
        best_match = None
        best_score = 0.0
        
        for endpoint in backend_endpoints:
            if endpoint['method'] != frontend_method:
                continue
            
            similarity = calculate_route_similarity(call['url'], endpoint['route'])
            
            if similarity > best_score and similarity > 0.7:
                best_score = similarity
                best_match = endpoint
        
        if best_match:
            mapping.append({
                "action_function": call['action_function'],
                "frontend_service": call['frontend_service'],
                "frontend_args": call['frontend_args'],
                "frontend_url": call['url'],
                "backend_function": best_match['function'],
                "backend_route": best_match['route'],
                "backend_method": best_match['method'],
                "backend_file": best_match['file'],
                "similarity_score": best_score
            })
    
    return mapping

# ----------------------------
# MAIN EXECUTION
# ----------------------------

def main():
    frontend_root = "../guided-workflow/src"
    backend_root = "./api"
    
    print(" Parsing frontend calls...")
    frontend_calls = parse_frontend_fixed(frontend_root)
    print(f" Found {len(frontend_calls)} frontend calls")
    
    print("🔍 Parsing backend endpoints...")
    backend_endpoints = parse_backend_with_prefixes_fixed(backend_root)
    print(f" Found {len(backend_endpoints)} backend endpoints")
    
    print(" Mapping frontend to backend...")
    api_mapping = map_frontend_to_backend_final(frontend_calls, backend_endpoints)
    print(f" Created {len(api_mapping)} mappings")
    
    # Save results
    with open("frontend_calls_final.json", "w") as f:
        json.dump(frontend_calls, f, indent=2)
    
    with open("backend_endpoints_final.json", "w") as f:
        json.dump(backend_endpoints, f, indent=2)
    
    with open("api_mapping_final.json", "w") as f:
        json.dump(api_mapping, f, indent=2)
    
    # Test specific cases
    print("\n Testing specific mappings:")
    
    # Look for createTagActions -> thought_spot_tag
    create_tag_actions = [m for m in api_mapping if m['action_function'] == 'createTagActions']
    if create_tag_actions:
        print(" createTagActions mapping:")
        print(json.dumps(create_tag_actions[0], indent=2))
    else:
        print(" createTagActions mapping not found")
        # Check if frontend call exists
        create_tag_frontend = [c for c in frontend_calls if c['action_function'] == 'createTagActions']
        if create_tag_frontend:
            print("Frontend call found:")
            print(json.dumps(create_tag_frontend[0], indent=2))
        
        # Check if backend endpoint exists
        thought_spot_backend = [e for e in backend_endpoints if 'thought_spot_tag' in e['route']]
        if thought_spot_backend:
            print("Backend endpoint found:")
            print(json.dumps(thought_spot_backend[0], indent=2))
    
    # Show mapping statistics
    print(f"\n Final Statistics:")
    print(f"Frontend calls: {len(frontend_calls)}")
    print(f"Backend endpoints: {len(backend_endpoints)}")
    print(f"Successful mappings: {len(api_mapping)}")
    print(f"Mapping rate: {len(api_mapping)/len(frontend_calls)*100:.1f}%")
    
    return frontend_calls, backend_endpoints, api_mapping

if __name__ == "__main__":
    frontend_calls, backend_endpoints, api_mapping = main()

 Parsing frontend calls...
 Found 145 frontend calls
🔍 Parsing backend endpoints...
 Found 82 backend endpoints
 Mapping frontend to backend...
 Created 73 mappings

 Testing specific mappings:
 createTagActions mapping:
{
  "action_function": "createTagActions",
  "frontend_service": "client.post",
  "frontend_args": "(`${V2_URL}/thought_spot_tag`, {\r\n    requests,\r\n  })",
  "frontend_url": "/thought_spot_tag",
  "backend_function": "tagging_task_runner",
  "backend_route": "/thought_spot_tag",
  "backend_method": "POST",
  "backend_file": "thought_spot_tag",
  "similarity_score": 1.0
}

 Final Statistics:
Frontend calls: 145
Backend endpoints: 82
Successful mappings: 73
Mapping rate: 50.3%


Progress So Far
Repo traversal and file ingestion – implemented directory walker, file filters, safe readers.


✅ Parsing with sqlglot – successful on standalone SQL statements :


Detected tables, CRUD ops, joins, columns.


Example: INSERT INTO orders, UPDATE users, DELETE logs, SELECT JOIN orders.

Where We’re Stuck
❌ Full stored procedure parsing:


sqlglot parses simple statements, but does not fully support procedure syntax (CREATE PROCEDURE ... BEGIN ... END).


Current output works for small SQLs but fails on larger production procedures (falls back to Command nodes).


❌ Procedural constructs unsupported:


DECLARE, BEGIN/END, control flow aren’t recognized as parseable nodes in sqlglot.


Tried AST doesn’t expose procedure internals.


What I Tried
Direct parsing with sqlglot AND

Experimented with AST traversal


Iterated over Insert, Update, Delete, Select → correct for simple Queries.


Larger bodies still collapse into opaque Command nodes.


Tried capturing blocks


Looked for exp.Block, but no such construct is in sqlglot.


In [ ]:
import sqlglot
from sqlglot import expressions as exp
import re

class SQLParser:
    def __init__(self):
        pass

    def parse(self, files: list[dict]) -> dict:
        results = []
        
        print(f"Total files to check: {len(files)}")
        
        for i, f in enumerate(files):
            file_path = f["file_path"]
            
            # Only process .sql files for now
            if not file_path.lower().endswith('.sql'):
                continue
            
            print(f"Processing SQL file: {file_path}")
            
            try:
                # SUPER AGGRESSIVE CONTENT CLEANING
                content = f["content"]
                
                # Remove ALL ANSI escape codes
                content = re.sub(r'\x1b\[[0-9;]*[mGKH]', '', content)
                
                # Remove comment blocks with # symbols
                content = re.sub(r'^#{3,}.*?#{3,}$', '', content, flags=re.MULTILINE)
                content = re.sub(r'^#{10,}.*$', '', content, flags=re.MULTILINE)
                
                # Remove lines that are just # characters
                lines = content.split('\n')
                cleaned_lines = []
                for line in lines:
                    # Skip lines that are mostly # characters
                    if line.strip() and not re.match(r'^#{3,}', line.strip()):
                        cleaned_lines.append(line)
                
                content = '\n'.join(cleaned_lines)
                
                # Split by semicolons (proper SQL statement separation)
                statements = content.split(';')
                
                for stmt_content in statements:
                    stmt_content = stmt_content.strip()
                    
                    # Skip empty or very short statements
                    if not stmt_content or len(stmt_content) < 15:
                        continue
                    
                    # Skip comment-only statements
                    if stmt_content.startswith('--') or stmt_content.startswith('#'):
                        continue
                    
                    # Only process statements that look like complete SQL
                    if not any(stmt_content.upper().strip().startswith(keyword) 
                              for keyword in ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'WITH']):
                        continue
                    
                    try:
                        # Try parsing with Snowflake dialect
                        tree = sqlglot.parse(stmt_content, dialect="snowflake")
                        

                        for stmt in tree:
                            if stmt:

                                # print(stmt)
                                result = {
                                    "file": file_path,
                                    "type": self._safe_get_type(stmt),
                                    "tables": self._safe_get_tables(stmt),
                                    "columns": self._safe_get_columns(stmt),
                                    "joins": self._safe_get_joins(stmt),
                                    "sql_preview": stmt_content[:150]
                                }
                                results.append(result)
                                
                    except Exception as stmt_error:
                        # Only log significant errors
                        error_msg = str(stmt_error)
                        if len(stmt_content) > 30 and "Unexpected token" in error_msg:
                            results.append({
                                "file": file_path,
                                "error": f"Parse error: {error_msg[:100]}",
                                "statement_preview": stmt_content[:80]
                            })
                        
            except Exception as e:
                results.append({
                    "file": file_path,
                    "error": f"File error: {str(e)}"
                })
        
        successful = len([r for r in results if 'error' not in r])
        errors = len([r for r in results if 'error' in r])
        
        print(f"\nSuccessfully parsed statements: {successful}")
        print(f"Parsing errors: {errors}")
        
        return {"sql": results}
    
    def _safe_get_type(self, stmt):
        try:
            return stmt.key.upper() if hasattr(stmt, 'key') else "UNKNOWN"
        except:
            return "UNKNOWN"
    
    def _safe_get_tables(self, stmt):
        try:
            tables = []
            for t in stmt.find_all(exp.Table):
                try:
                    table_name = str(t.name) if hasattr(t, 'name') else str(t)
                    table_name = table_name.replace('"', '').replace("'", '')
                    if table_name and table_name != 'UNKNOWN':
                        tables.append(table_name)
                except:
                    pass
            return tables
        except:
            return []
    
    def _safe_get_columns(self, stmt):
        try:
            columns = []
            for c in stmt.find_all(exp.Column):
                try:
                    col_name = str(c.name) if hasattr(c, 'name') else str(c)
                    col_name = col_name.replace('"', '').replace("'", '')
                    if col_name and col_name != 'UNKNOWN':
                        columns.append(col_name)
                except:
                    pass
            return columns
        except:
            return []
    
    def _safe_get_joins(self, stmt):
        try:
            joins = []
            for j in stmt.find_all(exp.Join):
                try:
                    join_type = "INNER"
                    if hasattr(j, 'args') and 'kind' in j.args:
                        join_type = str(j.args.get("kind", "INNER")).upper()
                    
                    right_table = None
                    if hasattr(j, 'this') and j.this:
                        right_table = str(j.this.name) if hasattr(j.this, 'name') else str(j.this)
                    
                    if right_table:
                        joins.append({
                            "type": join_type,
                            "table": right_table
                        })
                except:
                    pass
            return joins
        except:
            return []

# Test the improved parser
if __name__ == "__main__":
    sample = files[:5]
    parser = SQLParser()
    result = parser.parse(sample)
    
    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    
    successful = [r for r in result['sql'] if 'error' not in r]
    print(f"\nSuccessful parses: {len(successful)}")
    
    # Group by statement type
    by_type = {}
    for r in successful:
        stmt_type = r['type']
        if stmt_type not in by_type:
            by_type[stmt_type] = []
        by_type[stmt_type].append(r)
    
    print("\nStatement types found:")
    for stmt_type, statements in by_type.items():
        print(f"  {stmt_type}: {len(statements)} statements")
    import json
    
    print(json.dumps(by_type,indent =2))
    # Show sample of each type
    print("\nSample statements:")
    for stmt_type, statements in by_type.items():
        print(f"\n{stmt_type} Example:")
        r = statements[0]
        print(f"  Tables: {r['tables']}")
        print(f"  Columns: {r['columns'][:3]}...")
        if r['joins']:
            print(f"  Joins: {r['joins']}")
        print(f"  SQL: {r['sql_preview'][:100]}...")
    
    # Show remaining errors (should be much fewer)
    errors = [r for r in result['sql'] if 'error' in r]
    print(f"\nRemaining errors: {len(errors)}")

For the limitiations of the sqlglot i thought of using regex to crack the stored procedure structure that didnot help me this is the point am at. i tried asking claude to write a parser it tried using but that was not helpful it was mapping inaccurately.

In [ ]:
import csv
import uuid
import json
from datetime import datetime
from typing import List, Dict, Optional

class ComprehensiveSQLToCSVGenerator:
    def __init__(self):
        self.entities = []
        self.relationships = []
        self.operations = []
        self.relationship_types = []
        self.entity_lookup = {}
        
        self._init_relationship_types()
    
    def _init_relationship_types(self):
        """Initialize comprehensive relationship types"""
        types = [
            ("CONTAINS_TABLE", "STRUCTURE", "Schema contains table"),
            ("CONTAINS_COLUMN", "STRUCTURE", "Table contains column"),
            ("REFERENCES_TABLE", "USAGE", "Statement references table"),
            ("JOINS_WITH", "USAGE", "Table joins with another table"),
            ("INSERTS_INTO", "EXECUTION", "Statement inserts into table"),
            ("SELECTS_FROM", "USAGE", "Statement selects from table"),
            ("CREATES_TABLE", "EXECUTION", "Statement creates table"),
            ("UPDATES_TABLE", "EXECUTION", "Statement updates table"),
            ("DELETES_FROM", "EXECUTION", "Statement deletes from table"),
            ("USES_COLUMN", "USAGE", "Statement uses column"),
            ("DEFINES_COLUMN", "STRUCTURE", "Table defines column"),
        ]
        
        for name, category, description in types:
            self.relationship_types.append({
                'relationship_type_id': str(uuid.uuid4()),
                'name': name,
                'category': category,
                'description': description
            })
    
    def process_sql_results(self, sql_results: Dict):
        """Comprehensively process SQL parsing results"""
        
        for result in sql_results['sql']:
            if 'error' in result:
                continue
            
            file_path = result['file']
            stmt_type = result['type']
            tables = result['tables']
            columns = result['columns']
            joins = result.get('joins', [])
            sql_preview = result.get('sql_preview', '')
            
            # Create file entity
            file_entity_id = self._add_entity(
                entity_name=file_path.split('\\')[-1],
                entity_type="SQL_FILE",
                qualified_name=file_path,
                description=f"SQL file containing {stmt_type} statements"
            )
            
            # Process each table
            table_entities = []
            for table_name in tables:
                # Handle schema.table format
                schema_name = None
                if '.' in table_name:
                    parts = table_name.split('.')
                    if len(parts) >= 2:
                        schema_name = '.'.join(parts[:-1])
                        table_simple_name = parts[-1]
                    else:
                        table_simple_name = table_name
                else:
                    table_simple_name = table_name
                
                # Create schema entity if exists
                schema_entity_id = None
                if schema_name:
                    schema_entity_id = self._add_entity(
                        entity_name=schema_name,
                        entity_type="SCHEMA",
                        qualified_name=schema_name,
                        description=f"Database schema"
                    )
                
                # Create table entity
                table_entity_id = self._add_entity(
                    entity_name=table_simple_name,
                    entity_type="TABLE",
                    qualified_name=table_name,
                    description=f"Table referenced in {stmt_type} statement",
                    parent_entity_id=schema_entity_id
                )
                table_entities.append(table_entity_id)
                
                # Create schema contains table relationship
                if schema_entity_id:
                    self._add_relationship(
                        source_entity_id=schema_entity_id,
                        target_entity_id=table_entity_id,
                        relationship_type="CONTAINS_TABLE",
                        detail=f"Schema contains table {table_simple_name}"
                    )
                
                # Create statement-table relationships based on type
                if stmt_type == "INSERT":
                    self._add_relationship(
                        source_entity_id=file_entity_id,
                        target_entity_id=table_entity_id,
                        relationship_type="INSERTS_INTO",
                        detail=f"INSERT statement targets table"
                    )
                elif stmt_type == "SELECT":
                    self._add_relationship(
                        source_entity_id=file_entity_id,
                        target_entity_id=table_entity_id,
                        relationship_type="SELECTS_FROM",
                        detail=f"SELECT statement reads from table"
                    )
                elif stmt_type == "CREATE":
                    self._add_relationship(
                        source_entity_id=file_entity_id,
                        target_entity_id=table_entity_id,
                        relationship_type="CREATES_TABLE",
                        detail=f"CREATE statement defines table"
                    )
                else:
                    self._add_relationship(
                        source_entity_id=file_entity_id,
                        target_entity_id=table_entity_id,
                        relationship_type="REFERENCES_TABLE",
                        detail=f"{stmt_type} statement references table"
                    )
            
            # Process columns
            for column_name in columns:
                # Try to associate column with table
                parent_table_id = table_entities[0] if table_entities else None
                
                # Handle table.column format
                if '.' in column_name:
                    parts = column_name.split('.')
                    if len(parts) >= 2:
                        table_part = parts[0]
                        col_part = parts[-1]
                        # Find matching table entity
                        for table_name in tables:
                            if table_part.lower() in table_name.lower():
                                parent_table_id = self.entity_lookup.get(table_name)
                                break
                        column_name = col_part
                
                # Create column entity
                column_qualified_name = f"{tables[0] if tables else 'unknown'}.{column_name}"
                column_entity_id = self._add_entity(
                    entity_name=column_name,
                    entity_type="COLUMN",
                    qualified_name=column_qualified_name,
                    description=f"Column used in {stmt_type} statement",
                    parent_entity_id=parent_table_id
                )
                
                # Create table contains column relationship
                if parent_table_id:
                    self._add_relationship(
                        source_entity_id=parent_table_id,
                        target_entity_id=column_entity_id,
                        relationship_type="CONTAINS_COLUMN",
                        detail=f"Table contains column {column_name}"
                    )
                
                # Create statement uses column relationship
                self._add_relationship(
                    source_entity_id=file_entity_id,
                    target_entity_id=column_entity_id,
                    relationship_type="USES_COLUMN",
                    detail=f"{stmt_type} statement uses column"
                )
            
            # Process joins
            for i, join in enumerate(joins):
                if i < len(table_entities) - 1:
                    self._add_relationship(
                        source_entity_id=table_entities[i],
                        target_entity_id=table_entities[i + 1],
                        relationship_type="JOINS_WITH",
                        detail=f"{join['type']} join between tables"
                    )
            
            # Create comprehensive operation record
            operation_metadata = {
                "statement_type": stmt_type,
                "tables_count": len(tables),
                "columns_count": len(columns),
                "joins_count": len(joins),
                "file_path": file_path
            }
            
            self._add_operation(
                operation_type=stmt_type,
                source_entity_id=file_entity_id,
                target_entity_id=table_entities[0] if table_entities else None,
                sql_snippet=sql_preview,
                execution_context=file_path,
                metadata=operation_metadata
            )
    
    def _add_entity(self, entity_name: str, entity_type: str, qualified_name: str, 
                   description: str = "", parent_entity_id: Optional[str] = None) -> str:
        """Add entity with comprehensive metadata"""
        if qualified_name in self.entity_lookup:
            return self.entity_lookup[qualified_name]
        
        entity_id = str(uuid.uuid4())
        entity = {
            'entity_id': entity_id,
            'entity_name': entity_name,
            'entity_type': entity_type,
            'qualified_name': qualified_name,
            'parent_entity_id': parent_entity_id or '',
            'description': description,
            'metadata': json.dumps({
                "entity_type": entity_type,
                "created_by": "SQL_PARSER",
                "source": "automated_analysis"
            }),
            'created_at': datetime.now().isoformat(),
            'updated_at': datetime.now().isoformat()
        }
        
        self.entities.append(entity)
        self.entity_lookup[qualified_name] = entity_id
        return entity_id
    
    def _add_relationship(self, source_entity_id: str, target_entity_id: str, 
                         relationship_type: str, detail: str = ""):
        """Add relationship with metadata"""
        rel_type_id = None
        for rt in self.relationship_types:
            if rt['name'] == relationship_type:
                rel_type_id = rt['relationship_type_id']
                break
        
        if not rel_type_id:
            return
        
        relationship = {
            'relationship_id': str(uuid.uuid4()),
            'source_entity_id': source_entity_id,
            'target_entity_id': target_entity_id,
            'relationship_type_id': rel_type_id,
            'relationship_detail': detail,
            'metadata': json.dumps({
                "relationship_type": relationship_type,
                "created_by": "SQL_PARSER"
            }),
            'created_at': datetime.now().isoformat()
        }
        self.relationships.append(relationship)
    
    def _add_operation(self, operation_type: str, source_entity_id: str, 
                      target_entity_id: Optional[str], sql_snippet: str,
                      execution_context: str, metadata: Dict):
        """Add comprehensive operation record"""
        operation = {
            'operation_id': str(uuid.uuid4()),
            'operation_type': operation_type,
            'source_entity_id': source_entity_id,
            'target_entity_id': target_entity_id or '',
            'performed_at': datetime.now().isoformat(),
            'performed_by': 'SQL_PARSER',
            'sql_snippet': sql_snippet,
            'execution_context': execution_context,
            'metadata': json.dumps(metadata)
        }
        self.operations.append(operation)
    
    def export_to_csv(self, output_dir: str = "comprehensive_csv"):
        """Export comprehensive data to CSV files"""
        from pathlib import Path
        
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Export entities
        with open(output_path / "entities.csv", 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['entity_id', 'entity_name', 'entity_type', 'qualified_name',
                         'parent_entity_id', 'description', 'metadata', 'created_at', 'updated_at']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.entities)
        
        # Export operations  
        with open(output_path / "operations.csv", 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['operation_id', 'operation_type', 'source_entity_id', 'target_entity_id',
                         'performed_at', 'performed_by', 'sql_snippet', 'execution_context', 'metadata']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.operations)
        
        # Export relationships
        with open(output_path / "relationships.csv", 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['relationship_id', 'source_entity_id', 'target_entity_id',
                         'relationship_type_id', 'relationship_detail', 'metadata', 'created_at']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.relationships)
        
        # Export relationship types
        with open(output_path / "relationship_types.csv", 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['relationship_type_id', 'name', 'category', 'description']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.relationship_types)
        
        print(f"\n✅ Comprehensive CSV files generated in '{output_dir}/':")
        print(f"   📄 entities.csv ({len(self.entities)} records)")
        print(f"   📄 operations.csv ({len(self.operations)} records)")
        print(f"   📄 relationships.csv ({len(self.relationships)} records)")
        print(f"   📄 relationship_types.csv ({len(self.relationship_types)} records)")
        
        # Print summary
        entity_types = {}
        for entity in self.entities:
            et = entity['entity_type']
            entity_types[et] = entity_types.get(et, 0) + 1
        
        print(f"\n📊 Entity Summary:")
        for entity_type, count in entity_types.items():
            print(f"   • {entity_type}: {count}")

# Use this comprehensive version
if __name__ == "__main__":
    # Run your SQL parser
    sample = files[:5]
    parser = SQLParser()
    sql_results = parser.parse(sample)
    print(sql_results[0])
    # Generate comprehensive CSV files
    csv_generator = ComprehensiveSQLToCSVGenerator()
    csv_generator.process_sql_results(sql_results)
    csv_generator.export_to_csv("comprehensive_analysis")